In [1]:
import pyterrier as pt
import pandas as pd

dataset = pt.get_dataset("msmarco_passage")
index = dataset.get_index(variant="terrier_stemmed")

# Load DL-hard annotations
annotations_url = "https://raw.githubusercontent.com/grill-lab/DL-Hard/main/annotations/query/annotations.tsv"
annotations = pd.read_csv(
    annotations_url,
    sep="	",
    header=None,
    names=["qid", "query", "intent", "answer_type", "topic_domain", "serp_type"],
)
annotations["qid"] = annotations["qid"].astype(str)

# Load TREC DL 2019/2020 qrels
dl19 = pt.get_dataset("irds:msmarco-passage/trec-dl-2019")
dl20 = pt.get_dataset("irds:msmarco-passage/trec-dl-2020")
all_qrels = pd.concat([dl19.get_qrels(), dl20.get_qrels()])
all_qrels["qid"] = all_qrels["qid"].astype(str)

# Keep only queries that have both annotations and relevance judgements
qrels_qids = set(all_qrels["qid"].unique())
topics = annotations[annotations["qid"].isin(qrels_qids)].reset_index(drop=True)

print(f"Queries with both annotations and relevance values: {len(topics)}")
print(topics["intent"].value_counts())
print("")
print(topics["answer_type"].value_counts())

# Word-count based length grouping (standard in IR research).
# Character-based bins are too granular for ~97 queries and produce
# many near-empty groups that break statistical tests.
# Three balanced word-count groups give ~25-40 queries each.
topics["query_word_count"] = topics["query"].str.split().str.len()

def length_group(n):
    if n <= 3:
        return "Short (<=3 words)"
    elif n <= 6:
        return "Medium (4-6 words)"
    else:
        return "Long (7+ words)"

topics["query_length_group"] = topics["query_word_count"].apply(length_group)
print("Query length group distribution:")
print(topics["query_length_group"].value_counts())
print("Word count stats:")
print(topics["query_word_count"].describe())


Queries with both annotations and relevance values: 97
intent
description     48
list            10
quantity         9
reason           8
attribute        7
entity           5
verification     3
process          3
language         3
weather          1
Name: count, dtype: int64

answer_type
factoid              28
definition           21
long answer          16
list                 10
short description     7
comparison            5
short answer          5
guide                 2
weather               1
Long answer           1
multi-answer          1
Name: count, dtype: int64
Query length group distribution:
query_length_group
Medium (4-6 words)    57
Long (7+ words)       27
Short (<=3 words)     13
Name: count, dtype: int64
Word count stats:
count    97.000000
mean      5.752577
std       2.231549
min       2.000000
25%       4.000000
50%       5.000000
75%       7.000000
max      15.000000
Name: query_word_count, dtype: float64


In [2]:
import re

def clean_query(q):
    q = re.sub(r"[^\w\s]", " ", str(q))
    q = re.sub(r"\s+", " ", q).strip()
    return q

# Clean queries (else it gives errors in PyTerrier)
topics["query"] = topics["query"].apply(clean_query)

# Initialize BM25 retriever
bm25 = pt.terrier.Retriever(index, wmodel="BM25")

# Initialize classic query expansion models
rm3 = pt.rewrite.RM3(index)
bo1 = pt.rewrite.Bo1QueryExpansion(index)

bm25_rm3 = bm25 >> rm3 >> bm25
bm25_bo1 = bm25 >> bo1 >> bm25

Java started (triggered by Retriever.__init__) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


In [3]:
from pyterrier.measures import RR, nDCG, AP, R
import os

# --- Classical methods ---
results_classical = pt.Experiment(
    [bm25, bm25_rm3, bm25_bo1],
    topics[["qid", "query"]],
    all_qrels,
    eval_metrics=[nDCG @ 10, AP(rel=2), RR(rel=2) @ 10, R(rel=2) @ 100],
    names=["BM25", "RM3", "Bo1"],
    perquery=True,
)

# --- LLM-based methods ---
# Load pre-generated Query2Doc expansions
q2d_path = "query2doc/topics_with_q2d.csv"
q2d_topics = pd.read_csv(q2d_path)
q2d_topics["qid"] = q2d_topics["qid"].astype(str)
q2d_topics = q2d_topics.rename(columns={"q2d_query": "query"})
q2d_topics["query"] = q2d_topics["query"].apply(clean_query)

results_q2d = pt.Experiment(
    [bm25],
    q2d_topics[["qid", "query"]],
    all_qrels,
    eval_metrics=[nDCG @ 10, AP(rel=2), RR(rel=2) @ 10, R(rel=2) @ 100],
    names=["Q2D_Llama3"],
    perquery=True,
)

# Load pre-generated CoT expansions
cot_path = "cot/topics_with_cot.csv"
cot_topics = pd.read_csv(cot_path)
cot_topics["qid"] = cot_topics["qid"].astype(str)
cot_topics["cot_query"] = cot_topics["cot_query"].apply(clean_query)

results_cot = pt.Experiment(
    [bm25],
    cot_topics[["qid", "cot_query"]].rename(columns={"cot_query": "query"}),
    all_qrels,
    eval_metrics=[nDCG @ 10, AP(rel=2), RR(rel=2) @ 10, R(rel=2) @ 100],
    names=["CoT_Llama3"],
    perquery=True,
)

# Combine all results
results = pd.concat([results_classical, results_q2d, results_cot], ignore_index=True)

# Merge in query_length_group label (created in Cell 0)
label_map = topics[["qid", "query_length_group"]]
results = results.merge(label_map, on="qid", how="left")

# Overall results
print("=== Overall Results ===")
overall = results.groupby(["name", "measure"])["value"].mean().unstack()
print(overall)

# Results by query length
print("=== nDCG@10 by Query Length ===")
ndcg_by_length = (
    results[results["measure"] == "nDCG@10"]
    .groupby(["name", "query_length_group"])["value"]
    .mean()
    .unstack()
)
print(ndcg_by_length)
print("=== nDCG@10 by Query Length ===")
ndcg_by_length = (
    results[results["measure"] == "nDCG@10"]
    .groupby(["name", "query_length_group"])["value"]
    .mean()
    .unstack()
)
print(ndcg_by_length)


=== Overall Results ===
measure     AP(rel=2)  R(rel=2)@100  RR(rel=2)@10   nDCG@10
name                                                       
BM25         0.290089      0.541421      0.625749  0.487382
Bo1          0.311664      0.568762      0.618295  0.500851
CoT_Llama3   0.338962      0.559121      0.643941  0.514828
Q2D_Llama3   0.389330      0.628159      0.756554  0.578139
RM3          0.317579      0.575153      0.628371  0.516954
=== nDCG@10 by Query Length ===
query_length_group  Long (7+ words)  Medium (4-6 words)  Short (<=3 words)
name                                                                      
BM25                       0.510299            0.467334           0.527691
Bo1                        0.530557            0.478969           0.535098
CoT_Llama3                 0.502199            0.498091           0.614446
Q2D_Llama3                 0.553513            0.572956           0.652015
RM3                        0.548141            0.489776           0.571343

In [4]:
from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from itertools import combinations

# =============================================================
# STATISTICAL TESTS — QUERY LENGTH
# =============================================================
# Strategy:
#   - Kruskal-Wallis (omnibus): is there ANY difference across length groups?
#   - Pairwise Mann-Whitney U: which specific pairs (Short vs Medium, etc.) differ?
#   - Holm-Bonferroni correction across all pairwise p-values.
#
# Why Kruskal-Wallis instead of ANOVA? IR metric scores are bounded [0,1],
# skewed, and not normally distributed. Kruskal-Wallis is the non-parametric
# equivalent of one-way ANOVA and makes no distributional assumptions.
#
# Why Holm-Bonferroni? We run many tests simultaneously (5 methods x 4 metrics
# x 3 group pairs = 60 tests). Without correction, ~3 would appear significant
# by chance alone at p<0.05. Holm-Bonferroni controls this family-wise error.

def run_kruskal_pairwise(results_df, groupby_col, methods, measures):
    kruskal_rows = []
    pairwise_rows = []
    groups = results_df[groupby_col].dropna().unique()

    for method in methods:
        for measure in measures:
            subset = results_df[
                (results_df["name"] == method) & (results_df["measure"] == measure)
            ]
            group_scores = {
                g: subset[subset[groupby_col] == g]["value"].values
                for g in groups
            }
            valid_groups = [g for g, s in group_scores.items() if len(s) >= 2]
            if len(valid_groups) < 2:
                continue

            kw_stat, kw_p = kruskal(*[group_scores[g] for g in valid_groups])
            kruskal_rows.append({
                "method":       method,
                "measure":      measure,
                "kw_statistic": round(kw_stat, 4),
                "kw_p_value":   round(kw_p, 4),
                "group_means":  {g: round(group_scores[g].mean(), 4) for g in valid_groups},
            })

            for g1, g2 in combinations(valid_groups, 2):
                s1, s2 = group_scores[g1], group_scores[g2]
                stat, p = mannwhitneyu(s1, s2, alternative="two-sided")
                pairwise_rows.append({
                    "method":      method,
                    "measure":     measure,
                    "group_1":     g1,
                    "group_2":     g2,
                    "mw_stat":     round(stat, 4),
                    "p_value":     p,
                    "mean_group_1": round(s1.mean(), 4),
                    "mean_group_2": round(s2.mean(), 4),
                    "delta":       round(s1.mean() - s2.mean(), 4),
                })

    kruskal_df  = pd.DataFrame(kruskal_rows)
    pairwise_df = pd.DataFrame(pairwise_rows)

    if len(pairwise_df) > 0:
        reject, p_corr, _, _ = multipletests(pairwise_df["p_value"], method="holm")
        pairwise_df["p_corrected"] = p_corr.round(4)
        pairwise_df["significant"] = reject

    return kruskal_df, pairwise_df


methods  = results["name"].unique()
measures = results["measure"].unique()

kw_len, pw_len = run_kruskal_pairwise(results, "query_length_group", methods, measures)

print("=== Kruskal-Wallis: Query Length ===")
print(kw_len.to_string(index=False))

print("=== Pairwise Mann-Whitney (Holm-corrected): Query Length ===")
print(pw_len.sort_values("p_corrected").to_string(index=False))

print("=== Significant differences only (p_corrected < 0.05) ===")
if "significant" in pw_len.columns:
    sig = pw_len[pw_len["significant"]].sort_values("p_corrected")
    if len(sig) == 0:
        print("No significant pairwise differences found after Holm-Bonferroni correction.")
    else:
        print(sig.to_string(index=False))
else:
    print("No tests ran (check group sizes).")

# =============================================================
# METHOD vs. METHOD within each LENGTH GROUP
# =============================================================
# Answers: within Short queries, is Q2D significantly better than RM3?
# Same question for Medium and Long groups.
print("" + "="*60)
print("METHOD vs. METHOD within each Query Length Group")
print("="*60)

length_groups = results["query_length_group"].dropna().unique()
method_comp_rows = []

for group_label in length_groups:
    group_df = results[results["query_length_group"] == group_label]

    for measure in measures:
        measure_df = group_df[group_df["measure"] == measure]

        for m1, m2 in combinations(methods, 2):
            s1 = measure_df[measure_df["name"] == m1]["value"].values
            s2 = measure_df[measure_df["name"] == m2]["value"].values
            if len(s1) < 2 or len(s2) < 2:
                continue
            stat, p = mannwhitneyu(s1, s2, alternative="two-sided")
            method_comp_rows.append({
                "length_group": group_label,
                "measure":      measure,
                "method_1":     m1,
                "method_2":     m2,
                "mean_m1":      round(s1.mean(), 4),
                "mean_m2":      round(s2.mean(), 4),
                "delta":        round(s1.mean() - s2.mean(), 4),
                "mw_stat":      round(stat, 4),
                "p_value":      p,
            })

method_comp_df = pd.DataFrame(method_comp_rows)
reject, p_corr, _, _ = multipletests(method_comp_df["p_value"], method="holm")
method_comp_df["p_corrected"] = p_corr.round(4)
method_comp_df["significant"]  = reject

for grp in sorted(length_groups):
    print(f"--- {grp} ---")
    subset = (
        method_comp_df[method_comp_df["length_group"] == grp]
        .sort_values(["measure", "p_corrected"])
        [["measure","method_1","method_2","mean_m1","mean_m2","delta","p_value","p_corrected","significant"]]
    )
    print(subset.to_string(index=False))

print("=== Significant differences only (p_corrected < 0.05) ===")
sig = method_comp_df[method_comp_df["significant"]].sort_values(["length_group","measure","p_corrected"])
if len(sig) == 0:
    print("No significant pairwise method differences found after Holm-Bonferroni correction.")
else:
    print(sig[["length_group","measure","method_1","method_2","mean_m1","mean_m2","delta","p_corrected"]].to_string(index=False))


=== Kruskal-Wallis: Query Length ===
    method      measure  kw_statistic  kw_p_value                                                                            group_means
      BM25      nDCG@10        0.8677      0.6480 {'Medium (4-6 words)': 0.4673, 'Long (7+ words)': 0.5103, 'Short (<=3 words)': 0.5277}
      BM25    AP(rel=2)        1.3944      0.4980 {'Medium (4-6 words)': 0.2764, 'Long (7+ words)': 0.2706, 'Short (<=3 words)': 0.3903}
      BM25 R(rel=2)@100        1.1622      0.5593  {'Medium (4-6 words)': 0.552, 'Long (7+ words)': 0.4894, 'Short (<=3 words)': 0.6031}
      BM25 RR(rel=2)@10        0.3584      0.8359 {'Medium (4-6 words)': 0.6079, 'Long (7+ words)': 0.6716, 'Short (<=3 words)': 0.6085}
       Bo1      nDCG@10        0.7929      0.6727  {'Medium (4-6 words)': 0.479, 'Long (7+ words)': 0.5306, 'Short (<=3 words)': 0.5351}
       Bo1    AP(rel=2)        1.3399      0.5117 {'Medium (4-6 words)': 0.2954, 'Long (7+ words)': 0.2926, 'Short (<=3 words)': 0.4223}
    